# 01 — Data Collection

Download and assemble all raw datasets for the Pennsylvania healthcare access analysis.

**Data sources:**
- CMS Provider of Services (facility locations)
- HRSA Health Center data
- US Census American Community Survey (ACS) via API
- CDC Social Vulnerability Index (SVI)
- Census TIGER/Line shapefiles (tracts & roads)

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.config import DATA_RAW, STATE_FIPS, CENSUS_API_KEY
from src.data_collection import (
    download_cms_data,
    download_hrsa_data,
    fetch_acs_data,
    download_svi_data,
    download_tiger_shapefiles,
)

## 1.1 Healthcare Facility Data (CMS)

In [2]:
cms_df = download_cms_data(DATA_RAW)
cms_df.head(), cms_df.shape

Fetching PA hospital facilities from NPI Registry...


  Saved 639 hospital records to C:\Users\saksh\Desktop\DS440-Capstone-Project\data\raw\cms\cms_facilities_standardized.csv


(  facility_id               facility_name            address         city  \
 0  cms_000000            AARUSH MANCHANDA  100 N ACADEMY AVE     DANVILLE   
 1  cms_000001  ABINGTON MEMORIAL HOSPITAL     205 NEWTOWN RD   WARMINSTER   
 2  cms_000002  ABINGTON MEMORIAL HOSPITAL   1200 OLD YORK RD     ABINGTON   
 3  cms_000003  ABINGTON MEMORIAL HOSPITAL    7876 SPRING AVE  ELKINS PARK   
 4  cms_000004  ABINGTON MEMORIAL HOSPITAL   1200 OLD YORK RD     ABINGTON   
 
   state    zip  latitude  longitude  provider_count source  
 0    PA  17822       NaN        NaN             1.0    cms  
 1    PA  18974       NaN        NaN             1.0    cms  
 2    PA  19001       NaN        NaN             1.0    cms  
 3    PA  19027       NaN        NaN             1.0    cms  
 4    PA  19001       NaN        NaN             1.0    cms  ,
 (639, 10))

## 1.2 HRSA Health Centers

In [3]:
hrsa_df = download_hrsa_data(DATA_RAW)
hrsa_df.head(), hrsa_df.shape

Fetching PA health centers from NPI Registry...


  Saved 923 health center records to C:\Users\saksh\Desktop\DS440-Capstone-Project\data\raw\hrsa\hrsa_facilities_standardized.csv


(   facility_id                        facility_name                  address  \
 0  hrsa_000000  ALLIANCE COMMUNITY HEALTHCARE, INC.            1235 PENN AVE   
 1  hrsa_000001               B-K HEALTH CENTER, INC     25066 STATE ROUTE 11   
 2  hrsa_000002               B-K HEALTH CENTER, INC      498 S MAIN ST STE D   
 3  hrsa_000003               B-K HEALTH CENTER, INC  2380 ELK LAKE SCHOOL RD   
 4  hrsa_000004              B-K HEALTH CENTER, INC.            127 ROUTE 106   
 
                   city state    zip  latitude  longitude  provider_count  \
 0           WYOMISSING    PA  19610       NaN        NaN             1.0   
 1            HALLSTEAD    PA  18822       NaN        NaN             1.0   
 2             MONTROSE    PA  18801       NaN        NaN             1.0   
 3          SPRINGVILLE    PA  18844       NaN        NaN             1.0   
 4  GREENFIELD TOWNSHIP    PA  18407       NaN        NaN             1.0   
 
   source  
 0   hrsa  
 1   hrsa  
 2   hrsa  


## 1.3 Census ACS Demographics

In [4]:
if not CENSUS_API_KEY:
    raise ValueError("CENSUS_API_KEY is missing. Add it to .env or src/config.py")

acs_df = fetch_acs_data(state_fips=STATE_FIPS, year=2022, output_dir=DATA_RAW)
acs_df.head(), acs_df.shape

(                                              NAME  total_population  \
 0  Census Tract 301.01; Adams County; Pennsylvania              2658   
 1  Census Tract 301.03; Adams County; Pennsylvania              2416   
 2  Census Tract 301.04; Adams County; Pennsylvania              3395   
 3     Census Tract 302; Adams County; Pennsylvania              5475   
 4     Census Tract 303; Adams County; Pennsylvania              4412   
 
    median_household_income  insurance_universe  households  \
 0                    80231                2657        1061   
 1                   111603                2416         847   
 2                    81150                3395        1285   
 3                    68363                5451        2035   
 4                    81577                4412        1668   
 
    households_no_vehicle  race_total  race_white state county   tract  \
 0                      6        2658        2465    42    001  030101   
 1                     21       

## 1.4 CDC Social Vulnerability Index

In [5]:
_svi_path = DATA_RAW / "svi" / "SVI_2022_Pennsylvania.csv"
if _svi_path.exists():
    print("SVI data already downloaded — loading from disk.")
    svi_df = pd.read_csv(_svi_path)
else:
    svi_df = download_svi_data(state_fips=STATE_FIPS, output_dir=DATA_RAW)

svi_df.head(), svi_df.shape

SVI data already downloaded — loading from disk.


(   ST         STATE ST_ABBR  STCNTY        COUNTY         FIPS  \
 0  42  Pennsylvania      PA   42001  Adams County  42001030101   
 1  42  Pennsylvania      PA   42001  Adams County  42001030103   
 2  42  Pennsylvania      PA   42001  Adams County  42001030104   
 3  42  Pennsylvania      PA   42001  Adams County  42001030200   
 4  42  Pennsylvania      PA   42001  Adams County  42001030300   
 
                                           LOCATION  AREA_SQMI  E_TOTPOP  \
 0  Census Tract 301.01; Adams County; Pennsylvania  21.149362      2658   
 1  Census Tract 301.03; Adams County; Pennsylvania   6.963834      2416   
 2  Census Tract 301.04; Adams County; Pennsylvania  19.341957      3395   
 3     Census Tract 302; Adams County; Pennsylvania  46.744471      5475   
 4     Census Tract 303; Adams County; Pennsylvania  43.148038      4412   
 
    M_TOTPOP  ...  MP_AIAN  EP_NHPI  MP_NHPI  EP_TWOMORE  MP_TWOMORE  \
 0        14  ...      1.1      0.0      1.1         4.3         3

## 1.5 TIGER/Line Shapefiles (Tracts & Roads)

In [6]:
tiger_data = download_tiger_shapefiles(state_fips=STATE_FIPS, output_dir=DATA_RAW)
tracts_gdf = tiger_data["tracts"]
roads_gdf = tiger_data["roads"]

tracts_gdf.head(2), roads_gdf.head(2), (tracts_gdf.shape, roads_gdf.shape)

(  STATEFP COUNTYFP TRACTCE        GEOID               GEOIDFQ     NAME  \
 0      42      017  100102  42017100102  1400000US42017100102  1001.02   
 1      42      017  100103  42017100103  1400000US42017100103  1001.03   
 
                NAMELSAD  MTFCC FUNCSTAT    ALAND   AWATER     INTPTLAT  \
 0  Census Tract 1001.02  G5020        S  7695740  2422611  +40.0745577   
 1  Census Tract 1001.03  G5020        S  1578123    14537  +40.0682737   
 
        INTPTLON                                           geometry  
 0  -074.9405807  POLYGON ((-74.98425 40.0559, -74.98354 40.0566...  
 1  -074.9721495  POLYGON ((-74.98519 40.05711, -74.98515 40.057...  ,
        LINEARID            FULLNAME RTTYP  MTFCC  \
 0  110471840611  Kohler Mill Rd Exd     M  S1400   
 1  110471835209        Ridge Rd Exd     M  S1400   
 
                                             geometry  
 0  LINESTRING (-77.06289 39.85817, -77.06332 39.8...  
 1  LINESTRING (-77.25327 39.76478, -77.25327 39.7...  ,
 ((34

## 1.6 Initial Data Quality Report

In [7]:
quality_report = pd.DataFrame(
    [
        {"dataset": "CMS", "rows": len(cms_df), "cols": cms_df.shape[1]},
        {"dataset": "HRSA", "rows": len(hrsa_df), "cols": hrsa_df.shape[1]},
        {"dataset": "ACS", "rows": len(acs_df), "cols": acs_df.shape[1]},
        {"dataset": "SVI", "rows": len(svi_df), "cols": svi_df.shape[1]},
        {"dataset": "Tracts", "rows": len(tracts_gdf), "cols": tracts_gdf.shape[1]},
        {"dataset": "Roads", "rows": len(roads_gdf), "cols": roads_gdf.shape[1]},
    ]
)
quality_report

,dataset,rows,cols
0,CMS,639,10
1,HRSA,923,10
2,ACS,3446,12
3,SVI,3445,161
4,Tracts,3446,14
5,Roads,595975,5
